# Tideway Credentials API Examples

Raw `get`/`post`/`patch`/`delete` calls for vault credentials. Adjust endpoints/bodies per your appliance (see BMC docs).

## Setup
- Install tideway (e.g. `pip install -e .` from repo root).
- Set `TARGET` and `TOKEN` below. Do **not** commit secrets.

In [ ]:
import tideway

# Configuration
TARGET = 'appliance-hostname-or-ip'  # e.g. 'discovery.example.com'
TOKEN = 'your-api-token'            # keep secrets out of source control
API_VERSION = '1.16'                # latest supported API
SSL_VERIFY = False                  # set to True when using valid certs

tw = tideway.appliance(TARGET, TOKEN, api_version=API_VERSION, ssl_verify=SSL_VERIFY)
cred_id = None                      # set by the create cell or replace with an existing credential id

def show_response(resp, label):
    if resp.ok:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return {'endpoint': label, 'status': resp.status_code, 'body': body}

tw.api_about.json()  # quick sanity check


## Credential type endpoints (GET)

In [ ]:
# List credential types
cred_types = tw.get('/vault/credential_types')
show_response(cred_types, '/vault/credential_types')

# Get specific credential type (edit name)
cred_type_name = 'SNMPv2'  # example
cred_type = tw.get(f"/vault/credential_types/{cred_type_name}")
show_response(cred_type, f"/vault/credential_types/{cred_type_name}")


## Credential endpoints (GET/POST/PATCH/DELETE)

In [ ]:
# List credentials
creds = tw.get('/vault/credentials')
show_response(creds, '/vault/credentials')


In [ ]:

# Get a credential by id
if cred_id:
    cred = tw.get(f"/vault/credentials/{cred_id}")
    show_response(cred, f"/vault/credentials/{cred_id}")

In [ ]:

# Create a credential (edit body)
new_cred_body = {
    # 'name': 'example', 'type': 'SNMPv2', 'attributes': {...}
}
if new_cred_body:
    created = tw.post('/vault/credentials', new_cred_body)
    show_response(created, '/vault/credentials (POST)')
    cred_id = created.json().get('id') if created.ok else None
else:
    cred_id = None


In [ ]:

# Patch a credential (edit body)
patch_body = {
    # 'attributes': {'community': 'public'}
}
if cred_id and patch_body:
    patched = tw.patch(f"/vault/credentials/{cred_id}", patch_body)
    show_response(patched, f"/vault/credentials/{cred_id} (PATCH)")


In [ ]:

# Delete a credential
if cred_id:
    deleted = tw.delete(f"/vault/credentials/{cred_id}")
    show_response(deleted, f"/vault/credentials/{cred_id} (DELETE)")
